# Clase 188 — Inferencia causal: DAGs, confounders, instrumentos

Correlación ≠ causalidad. Simulamos un DAG con numpy y mostramos: el sesgo por **confounder** omitido y su corrección por ajuste, el daño de controlar un **collider**, y la recuperación del efecto con **variables instrumentales (2SLS)** cuando hay confounders no observados.

Requiere: `numpy`, `pandas`, `statsmodels`, `matplotlib`.

## 🧠 Intuición previa

**Causalidad ≠ correlación.** Que dos cosas se muevan juntas no dice cuál causa a cuál (ni si un tercero causa a ambas). Un **DAG** (grafo dirigido acíclico) dibuja *qué causa qué*: así sabés **qué variables controlar** (los *confounders*, que abren caminos espurios) y, crucialmente, **cuáles NO** (los *colliders* y *mediators*, que si los controlás introducen sesgo en vez de quitarlo).

## 1. Confounder (fork): sesgo por variable omitida

`T ← Z → Y`. `Z` causa tanto `T` como `Y`. El OLS ingenuo `Y~T` está sesgado; ajustar por `Z` recupera el efecto causal (backdoor criterion).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.sandbox.regression.gmm import IV2SLS
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

n = 5000
Z = rng.normal(0, 1, n)                     # confounder
T = 0.8 * Z + rng.normal(0, 1, n)           # tratamiento depende de Z
Y = 2.0 * T + 3.0 * Z + rng.normal(0, 1, n) # efecto causal verdadero de T = 2.0
df = pd.DataFrame({"Y": Y, "T": T, "Z": Z})
naive = smf.ols("Y ~ T", data=df).fit().params["T"]
adj   = smf.ols("Y ~ T + Z", data=df).fit().params["T"]
print("efecto verdadero = 2.00")
print(f"OLS ingenuo   Y~T   = {naive:.3f}  (sesgado por Z)")
print(f"OLS ajustado  Y~T+Z = {adj:.3f}  (recupera el efecto)")
assert abs(adj - 2.0) < 0.15 and abs(naive - 2.0) > 0.3

## 2. Collider: controlar de más introduce sesgo

`T → C ← Y`. Controlar el collider `C` **crea** una asociación espuria y destruye la estimación. Controlar "todo lo que tengo" es un error.

In [ ]:
T2 = rng.normal(0, 1, n)
Y2 = rng.normal(T2, 1)                 # Y depende de T (efecto 1.0)
C = T2 + Y2 + rng.normal(0, 1, n)      # collider: hijo de T y de Y
d2 = pd.DataFrame({"Y": Y2, "T": T2, "C": C})
b_free = smf.ols("Y ~ T", data=d2).fit().params["T"]
b_ctrl = smf.ols("Y ~ T + C", data=d2).fit().params["T"]
print(f"sin controlar C   Y~T   = {b_free:.3f}  (≈1, correcto)")
print(f"controlando C     Y~T+C = {b_ctrl:.3f}  (sesgado: controlar el collider DAÑA)")
assert abs(b_free - 1.0) < 0.1 and b_ctrl < b_free

## 3. Variables instrumentales (2SLS)

Con un confounder **no observado** `U` entre `T` e `Y`, el OLS es inconsistente. Un instrumento `Z` (afecta a `T`, no a `Y` salvo vía `T`) identifica el efecto con 2SLS.

In [ ]:
U = rng.normal(0, 1, n)                          # confounder NO observado
Zi = rng.normal(0, 1, n)                         # instrumento
Ti = 0.7 * Zi + 1.0 * U + rng.normal(0, 1, n)
Yi = 1.5 * Ti + 2.0 * U + rng.normal(0, 1, n)    # efecto verdadero 1.5
dta = pd.DataFrame({"Y": Yi, "T": Ti, "Z": Zi})

ols_biased = smf.ols("Y ~ T", data=dta).fit().params["T"]
exog  = sm.add_constant(dta[["T"]])
instr = sm.add_constant(dta[["Z"]])
iv = IV2SLS(dta["Y"], exog, instrument=instr).fit()
first = smf.ols("T ~ Z", data=dta).fit()
print("efecto verdadero = 1.50")
print(f"OLS (U no observado) = {ols_biased:.3f}  (sesgado)")
print(f"IV 2SLS              = {iv.params['T']:.3f}  (recupera ~1.5)")
print(f"F 1ª etapa = {first.fvalue:.1f}  (>10 => instrumento fuerte, regla de Stock-Yogo)")
assert abs(iv.params["T"] - 1.5) < 0.2 and first.fvalue > 10

## 4. Resumen visual del sesgo

Comparamos cada estimador contra su valor verdadero.

In [ ]:
labels = ["ingenuo\n(confounder)", "ajustado\n(backdoor)", "OLS\n(U no obs)", "IV 2SLS"]
vals   = [naive, adj, ols_biased, iv.params["T"]]
truths = [2.0, 2.0, 1.5, 1.5]
xpos = np.arange(len(labels))
plt.figure(figsize=(7, 4))
plt.bar(xpos, vals, color="steelblue", label="estimado")
plt.plot(xpos, truths, "rD", ms=11, label="verdadero")
plt.xticks(xpos, labels); plt.legend()
plt.title("Sesgo por confounders y su corrección")
plt.tight_layout(); plt.show()

## Ejercicios

1. Repetí el bloque 1 pero controlando además una variable irrelevante `W ~ N(0,1)`: verificá que no cambia el efecto estimado.
2. Debilitá el instrumento (`Ti = 0.05·Zi + U + ε`) y observá cómo la F de la 1ª etapa cae debajo de 10 y el 2SLS se vuelve inestable.
3. Simulá un mediator `T → M → Y` y mostrá que controlar `M` sesga el efecto total hacia 0.

## Conclusiones

- Dibujá el DAG **antes** de decidir qué controlar: no todo control mejora la estimación.
- Confounder (fork) → ajustar; collider → NO ajustar; mediator → ajustar bloquea el efecto indirecto.
- El OLS es causal solo bajo unconfoundedness; con confounders no observados usá IV, DiD o RDD.
- Un instrumento débil (F < 10) produce estimaciones sesgadas e inestables.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y **ejecutables** de los ejercicios del README (sección `## 🧪 Ejercicios`). Datos sintéticos con `np.random.default_rng(42)`, sin internet. Cada bloque imprime resultados y valida con `assert`.

### Ejercicio 1 — DAG en código
DAG con T, Y, Z (confounder) y C (collider). Identificar el adjustment set por el backdoor criterion (sin `dowhy`/`networkx` obligatorios).

In [ ]:
import numpy as np
# DAG: Z->T, Z->Y (confounder fork), T->Y (efecto causal), T->C<-Y (collider)
edges = [("Z","T"), ("Z","Y"), ("T","Y"), ("T","C"), ("Y","C")]
parents = {}
for a, b in edges:
    parents.setdefault(b, []).append(a)
print("Aristas:", edges)
print("Padres :", parents)
# Backdoor T->Y: el unico camino por la puerta trasera es T<-Z->Y => controlar {Z}.
# NO controlar C (collider T->C<-Y): abrirlo introduce sesgo de seleccion.
adjustment_set = {"Z"}
colliders = {"C"}
print(f"Backdoor adjustment set (Pearl) = {adjustment_set}")
print(f"NO controlar (collider)         = {colliders}")
assert "Z" in adjustment_set and "C" not in adjustment_set
try:
    import networkx as nx
    G = nx.DiGraph(edges)
    print("networkx: es DAG valido ->", nx.is_directed_acyclic_graph(G))
except ImportError:
    print("networkx no instalado; DAG representado como lista de aristas.")

### Ejercicio 2 — Sesgo del collider
`T ~ N(0,1)`, `Y ~ N(T,1)`, `C = T + Y + eps`. Controlar el collider C **destruye** la relación causal.

In [ ]:
import statsmodels.api as sm
rng = np.random.default_rng(42)
n = 5000
T = rng.normal(0, 1, n)
Y = T + rng.normal(0, 1, n)                 # efecto causal T->Y = 1
C = T + Y + rng.normal(0, 1, n)             # collider
b_free = sm.OLS(Y, sm.add_constant(T)).fit().params[1]
b_ctrl = sm.OLS(Y, sm.add_constant(np.column_stack([T, C]))).fit().params[1]
print(f"coef T sin controlar C (correcto ~1) = {b_free:.3f}")
print(f"coef T controlando C (SESGADO)       = {b_ctrl:.3f}")
assert abs(b_free - 1.0) < 0.15 and b_ctrl < b_free - 0.3

### Ejercicio 3 — Backdoor ajustando el confounder
`Z`, `T = f(Z)+eps`, `Y = 2T + 3Z + delta`. `OLS(Y~T)` está sesgado; `OLS(Y~T+Z)` recupera el 2.

In [ ]:
rng = np.random.default_rng(42)
n = 5000
Z = rng.normal(0, 1, n)
T = 1.5*Z + rng.normal(0, 1, n)             # Z afecta a T
Y = 2.0*T + 3.0*Z + rng.normal(0, 1, n)     # efecto causal = 2; Z confounder
b_naive = sm.OLS(Y, sm.add_constant(T)).fit().params[1]
b_adj = sm.OLS(Y, sm.add_constant(np.column_stack([T, Z]))).fit().params[1]
print(f"OLS Y~T   (sesgado)        = {b_naive:.3f}")
print(f"OLS Y~T+Z (backdoor, ~2)   = {b_adj:.3f}")
assert abs(b_adj - 2.0) < 0.1 and b_naive > 2.3

### Ejercicio 4 — Variables instrumentales (2SLS)
`linearmodels` no está instalada → 2SLS a mano con dos OLS. Instrumento `Z→T→Y` con confounder no observado `U`. Verificar F de 1ª etapa ≥ 10.

In [ ]:
rng = np.random.default_rng(42)
n = 5000
U = rng.normal(0, 1, n)                      # confounder NO observado
Z = rng.normal(0, 1, n)                      # instrumento
T = 0.8*Z + U + rng.normal(0, 1, n)          # Z relevante; U confunde
Y = 2.0*T + 3.0*U + rng.normal(0, 1, n)      # efecto causal = 2
b_ols = sm.OLS(Y, sm.add_constant(T)).fit().params[1]     # ingenuo, sesgado por U
first = sm.OLS(T, sm.add_constant(Z)).fit()               # 1a etapa
T_hat = first.fittedvalues
b_2sls = sm.OLS(Y, sm.add_constant(T_hat)).fit().params[1]  # 2a etapa
print(f"OLS ingenuo (sesgado por U) = {b_ols:.3f}")
print(f"2SLS (~2)                    = {b_2sls:.3f}")
print(f"F 1a etapa (regla >10)       = {first.fvalue:.1f}")
assert abs(b_2sls - 2.0) < 0.2 and first.fvalue > 10

### Ejercicio 5 — Double Machine Learning
`doubleml` no está instalada → DML (partialling-out) a mano con Random Forest y cross-fitting. Confounders no lineales; DML es el menos sesgado.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
rng = np.random.default_rng(42)
n = 3000
X = rng.normal(0, 1, (n, 3))
gX = np.sin(X[:,0]) + X[:,1]**2 + 0.5*X[:,2]     # confounder NO lineal en Y
mX = 0.5*np.cos(X[:,0]) + 0.3*X[:,1]              # propension no lineal
T = mX + rng.normal(0, 1, n)
Y = 2.0*T + gX + rng.normal(0, 1, n)             # efecto causal = 2

def dml_plr(X, T, Y, n_folds=5, seed=42):
    kf = KFold(n_folds, shuffle=True, random_state=seed)
    Tr = np.zeros_like(T); Yr = np.zeros_like(Y)
    for tr, te in kf.split(X):
        mt = RandomForestRegressor(n_estimators=100, random_state=seed).fit(X[tr], T[tr])
        my = RandomForestRegressor(n_estimators=100, random_state=seed).fit(X[tr], Y[tr])
        Tr[te] = T[te] - mt.predict(X[te])       # residuos (Neyman-orthogonal)
        Yr[te] = Y[te] - my.predict(X[te])
    return (Tr @ Yr) / (Tr @ Tr)

theta_dml = dml_plr(X, T, Y)
b_ols_lin = sm.OLS(Y, sm.add_constant(np.column_stack([T, X]))).fit().params[1]
print(f"OLS lineal (sesgado, g no lineal) = {b_ols_lin:.3f}")
print(f"DoubleML RF (~2)                  = {theta_dml:.3f}")
assert abs(theta_dml - 2.0) < 0.25